In [28]:
%load_ext autoreload
%autoreload 2

from meles.recognizer.arcface import ArcFaceRecognizer
from meles.aggregation.aggregator import NoAggregator, Aggregator
from meles.aggregation.pooling import MeanPoolingAggregator
from meles.classifier import Classifier
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Allow the usage of progress bars on pandas dataframes
tqdm.pandas()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
# Load the dataset from the experiment structure file.
DATA_PATH = Path("../../data")

experiment = pd.read_json(DATA_PATH.joinpath("ytfaces_experiment.json"))
experiment

,identity,video,frame,frame_suffix,path
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.555.jpg
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.556.jpg
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.557.jpg
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.558.jpg
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.559.jpg
...,...,...,...,...,...
130722,Zulfiqar_Ahmed,4,aligned_detect_4.225.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.225.jpg
130723,Zulfiqar_Ahmed,4,aligned_detect_4.226.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.226.jpg
130724,Zulfiqar_Ahmed,4,aligned_detect_4.227.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.227.jpg
130725,Zulfiqar_Ahmed,4,aligned_detect_4.228.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.228.jpg


In [30]:
# Keep the first three videos of every identity in the train set and move the rest into the test set.
TRAIN_VIDEOS_PER_IDENTITY = 3

# Rank the distinct videos of each identity by their natural (ascending) order, so
# that the three lowest-numbered videos of every identity receive ranks 1, 2 and 3.
video_rank = experiment.groupby("identity")["video"].transform(
    lambda videos: videos.rank(method="dense")
)
is_train = video_rank <= TRAIN_VIDEOS_PER_IDENTITY

train = experiment[is_train].reset_index(drop=True)
test = experiment[~is_train].reset_index(drop=True)

# Every identity keeps at least its first video in the train set, so all test identities
# are guaranteed to be present in the train set as well.
assert set(test["identity"]).issubset(set(train["identity"]))

print(f"Train: {len(train):>8,} frames | {train.groupby(['identity', 'video']).ngroups:>5,} videos | {train['identity'].nunique():>4,} identities")
print(f"Test:  {len(test):>8,} frames | {test.groupby(['identity', 'video']).ngroups:>5,} videos | {test['identity'].nunique():>4,} identities")

Train:  110,367 frames | 1,599 videos |  533 identities
Test:    20,360 frames |   293 videos |  226 identities


In [31]:
# For every identity, embed all frames into a list of embeddings (with potentially different size).
def embed(agg: Aggregator, data: pd.DataFrame) -> pd.DataFrame:
    embeddings = (
        data.groupby('identity')['path']
        .progress_apply(lambda paths: agg.embed([str(DATA_PATH / path) for path in paths]))
        .explode()
        .rename('embedding')
        .reset_index()
        .dropna(subset=['embedding'])
    )
    return embeddings

In [32]:
# Initialize and configure the recognizer, aggregator and classifier. The classifier has to use the metric of the recognizer.
aggregator = MeanPoolingAggregator(ArcFaceRecognizer())
classifier = Classifier(metric=aggregator.metric())

In [33]:
train_embeddings = embed(aggregator, train)
train_embeddings.to_json(DATA_PATH / f"ytfaces_embeddings_train_{aggregator.name()}.json")
train_embeddings

 12%|█▏        | 63/533 [08:28<1:03:10,  8.07s/it]


KeyboardInterrupt: 

In [ ]:
test_embeddings = embed(aggregator, test)
test_embeddings.to_json(DATA_PATH / f"ytfaces_embeddings_test_{aggregator.name()}.json")
test_embeddings

## Fit and evaluate the classifier

Fit the classifier on the train embeddings using the identity as the label, then classify every
test frame against the fitted gallery. We report the top-1 **accuracy** and the **rank-1** metric.
Since the classifier assigns each probe the identity of its single closest gallery neighbour, the
top-1 accuracy equals the rank-1 identification rate; the additional ranks show how quickly the
correct identity is recovered when more neighbours are considered.

In [ ]:
# Fit the classifier on the train embeddings, using the identity as the label.
def to_matrix(embeddings: pd.DataFrame) -> np.ndarray:
    """Stack the per-frame embedding lists into a (num_frames, embedding_dim) matrix."""
    return np.vstack(embeddings["embedding"].to_numpy())

X_train = to_matrix(train_embeddings)
y_train = train_embeddings["identity"].to_numpy()

classifier.fit(X_train, y_train)

In [ ]:
# Classify every test frame against the fitted gallery and evaluate the classifier.
from sklearn.metrics import accuracy_score

X_test = to_matrix(test_embeddings)
y_test = test_embeddings["identity"].to_numpy()

# Retrieve the ranked gallery neighbours (closest first) for every test frame.
RANKS = (1, 5, 10)
neighbor_labels, _ = classifier.classify(X_test, n_neighbors=max(RANKS))

# Top-1 accuracy: the closest neighbour's identity is the predicted identity.
y_pred = neighbor_labels[:, 0]
accuracy = accuracy_score(y_test, y_pred)

# Rank-k identification rate: the true identity appears within the k closest neighbours.
def rank_k_rate(k: int) -> float:
    hits = (neighbor_labels[:, :k] == y_test[:, None]).any(axis=1)
    return float(hits.mean())

print(f"Test frames: {len(y_test):,}")
print(f"Accuracy:    {accuracy:.4f}")
for k in RANKS:
    print(f"Rank-{k:<2}      {rank_k_rate(k):.4f}")

NoAggregator
Test frames: 20,360
Accuracy:    0.6344
Rank-1       0.6344
Rank-5       0.6809
Rank-10      0.7020

MeanPoolingAggregator
Test frames: 226
Accuracy:    0.5973
Rank-1       0.5973
Rank-5       0.6460
Rank-10      0.6770